# The local Kummer image at $p = 2$ for $y^2 = x^3 + 1$

For $E : y^2 = x^3+1$ (curve `36a1`, $j = 0$, CM by $\mathbb{Z}[\zeta_3]$) we write down an explicit
finite set $V \subset E(\mathbb{Q}_2)$ representing $E(\mathbb{Q}_2)/2E(\mathbb{Q}_2)$, and compute
its image under the local Kummer map at $2$ as explicit elements of
$$\mathbb{Q}_4 \;=\; \mathbb{Q}_2(\zeta_3) \;=\; \mathbb{Q}_2(\sqrt{-3}) \;=\; \mathbb{Q}_2(\sqrt 5),$$
the unramified quadratic extension of $\mathbb{Q}_2$.

**The answer.** With $\omega = \zeta_3$ and $\zeta_6 = -\omega^2 = 1+\omega$, the map is
$\delta(x,y) = x - \zeta_6 = (x-1) - \omega$, and
$$V = \{\mathcal{O},\ (-1,0),\ (-2,y),\ (-4,-3y)\}, \qquad y = \sqrt{-7} \in \mathbb{Z}_2^\times,\ y \equiv 3 \ (8),$$
$$\operatorname{Im}(\delta) = \{1,\ -2-\omega,\ -3-\omega,\ -5-\omega\}\cdot(\mathbb{Q}_4^\times)^2
 = \big\langle [-2-\omega],\, [-3-\omega] \big\rangle \subset \mathbb{Q}_4^\times/(\mathbb{Q}_4^\times)^2 .$$
The four classes are distinct for a one-line reason: their norms to $\mathbb{Q}_2$ are $1, 3, 7, 21$,
i.e. $1,3,7,5$ mod $8$ — *all four* square classes of $\mathbb{Z}_2^\times$ — and a square has a
square norm.

Two engines are used below, deliberately: Sage's `Qq(4)` for honest $2$-adic arithmetic and
square testing, and the number field $\mathbb{Q}(\omega)$ with `hilbert_symbol` at its inert prime
above $2$ (whose completion *is* $\mathbb{Q}_4$) for the pairing computations of §7. Everything
displayed symbolically is exact; only squareness is decided $2$-adically.

Companion to `local-kummer-p2.typ` in the **kummer** repo, which proves what is checked here;
the pari version of these computations is `local-kummer-p2.gp` there, with its transcript in
`results/local-kummer-p2.txt`.

## 1. The curve at $2$, and the size of $E(\mathbb{Q}_2)/2E(\mathbb{Q}_2)$

In [1]:
E = EllipticCurve([0,0,0,0,1])              # y^2 = x^3 + 1
print("E             :", E)
print("discriminant  :", E.discriminant(), "=", factor(E.discriminant()))
print("j-invariant   :", E.j_invariant(), "   conductor:", E.conductor())
print("globally minimal:", E.is_minimal())

ld = E.local_data(2)
print("at p = 2      : Kodaira", ld.kodaira_symbol(),
      "  f =", ld.conductor_valuation(), "  c_2 =", ld.tamagawa_number())
print("E(Q)_tors     :", E.torsion_subgroup().invariants(),
      "  generated by", [T.xy() for T in E.torsion_points() if T.order() == 6][:1])

E             : Elliptic Curve defined by y^2 = x^3 + 1 over Rational Field
discriminant  : -432 = -1 * 2^4 * 3^3
j-invariant   : 0    conductor: 36
globally minimal: True
at p = 2      : Kodaira IV   f = 2   c_2 = 3
E(Q)_tors     : (6,)   generated by [(2, -3)]


In [2]:
R.<X> = QQ[]
print("x^3 + 1 over Q :", factor(X^3 + 1))
print("-3 mod 8       :", Mod(-3, 8), "  so -3 is a square in Q_2 ?", Qp(2,20)(-3).is_square())
print("=> x^2 - x + 1 stays irreducible over Q_2, and E(Q_2)[2] = {O, (-1,0)}, r_2 = 1")
print("=> #E(Q_2)/2E(Q_2) = #E(Q_2)[2] * |2|_2^(-1) = 2 * 2 = 4 = 2^(r_2+1)")

x^3 + 1 over Q : (X + 1) * (X^2 - X + 1)
-3 mod 8       : 5   so -3 is a square in Q_2 ? False
=> x^2 - x + 1 stays irreducible over Q_2, and E(Q_2)[2] = {O, (-1,0)}, r_2 = 1
=> #E(Q_2)/2E(Q_2) = #E(Q_2)[2] * |2|_2^(-1) = 2 * 2 = 4 = 2^(r_2+1)


Type $\mathrm{IV}$ with $c_2 = 3$ pins the group down completely:
$E(\mathbb{Q}_2)/E_0 \cong \mathbb{Z}/3$, $E_0/E_1 \cong \tilde E_{\mathrm{ns}}(\mathbb{F}_2)
= (\mathbb{F}_2,+)$, and $E_1 \cong \mathbb{Z}_2$ — torsion free, because the only $2$-torsion point
$(-1,0)$ reduces to the *smooth* point $(1,0)$ (the singular point of $\tilde E$ is $(0,1)$). So
$$E(\mathbb{Q}_2) \cong \mathbb{Z}_2 \oplus \mathbb{Z}/6, \qquad
E(\mathbb{Q}_2)/2E(\mathbb{Q}_2) \cong (\mathbb{Z}/2)^2 .$$

This already shapes the search for $V$: **the global points are not enough.** $E(\mathbb{Q}) =
\mathbb{Z}/6$ covers only two of the four cosets — $(0,\pm1)$ is $3$-torsion, hence in
$2E(\mathbb{Q}_2)$, and $(2,\pm3) = T_0 + (0,\mp1)$ sits in the coset of $T_0$. The other two cosets
need a point with irrational $y$.

**The map.** Over $\mathbb{Q}_2$ the Galois action on $E[2]$ fixes $T_0 = (-1,0)$ and swaps the two
conjugate points, so $E[2] = \operatorname{Ind}_{\mathbb{Q}_4}^{\mathbb{Q}_2}\mu_2$ is free of rank
one over $\mathbb{F}_2[\operatorname{Gal}(\mathbb{Q}_4/\mathbb{Q}_2)]$ and Shapiro gives
$H^1(\mathbb{Q}_2, E[2]) \cong \mathbb{Q}_4^\times/(\mathbb{Q}_4^\times)^2$ — the target is one
field's multiplicative group, not an étale algebra. Concretely: the classical descent map into
$L = \mathbb{Q}_2[x]/(x^3+1) = \mathbb{Q}_2 \times \mathbb{Q}_4$ lands in the norm-one subgroup, and
since $(x+1)\cdot N_{\mathbb{Q}_4/\mathbb{Q}_2}(x - \zeta_6) = x^3+1 = y^2$, the
$\mathbb{Q}_2$-entry is determined mod squares by the $\mathbb{Q}_4$-entry. Nothing is lost by
keeping only
$$\delta(x,y) = x - \zeta_6 = (x-1) - \omega , \qquad \delta(\mathcal{O}) = 1 .$$

## 2. The field $\mathbb{Q}_4$, and which classes are squares

In [3]:
K2.<w> = Qq(4, 30)                 # unramified quadratic extension of Q_2; w = zeta_3
print("K2            :", K2)
print("residue field :", K2.residue_field(), "   e =", K2.e(), "  f =", K2.f())
print("w^2 + w + 1   =", w^2 + w + 1)
print("(1 + 2w)^2    =", (1 + 2*w)^2, "     so sqrt(-3) = 1 + 2w")
print()
for c in [-1, 2, 5, -3, 3, 7, 21]:
    print("  [" + str(c).rjust(3) + "] a square in Q_4 ?  " + str(K2(c).is_square()))

K2            : 2-adic Unramified Extension Field in w defined by x^2 + x + 1
residue field : Finite Field in w0 of size 2^2    e = 1   f = 2
w^2 + w + 1   = O(2^30)
(1 + 2w)^2    = 1 + 2^2 + 2^3 + 2^4 + 2^5 + 2^6 + 2^7 + 2^8 + 2^9 + 2^10 + 2^11 + 2^12 + 2^13 + 2^14 + 2^15 + 2^16 + 2^17 + 2^18 + 2^19 + 2^20 + 2^21 + 2^22 + 2^23 + 2^24 + 2^25 + 2^26 + 2^27 + 2^28 + 2^29 + O(2^30)      so sqrt(-3) = 1 + 2w

  [ -1] a square in Q_4 ?  False
  [  2] a square in Q_4 ?  False
  [  5] a square in Q_4 ?  True
  [ -3] a square in Q_4 ?  True
  [  3] a square in Q_4 ?  False
  [  7] a square in Q_4 ?  False
  [ 21] a square in Q_4 ?  True


So $5$ *is* a square in $\mathbb{Q}_4$ (it is unramified, $\mathbb{Q}_4 = \mathbb{Q}_2(\sqrt5)$),
and for $a \in \mathbb{Q}_2^\times$: $a$ is a square in $\mathbb{Q}_4$ exactly when $v_2(a)$ is even
and its unit part is $\equiv 1$ or $5 \pmod 8$. In particular $[3] = [7] = [-1]$ and $[21] = [1]$,
even though $3, 7, 21$ are non-squares in $\mathbb{Q}_2$.

### Why $\#\mathbb{Q}_4^\times/(\mathbb{Q}_4^\times)^2 = 2^{[\mathbb{Q}_4:\mathbb{Q}_2]+2} = 16$

For any finite $K/\mathbb{Q}_2$ of degree $d$ with residue field of size $q$,
$$K^\times \cong \pi^{\mathbb{Z}} \times \mu_{q-1} \times U^{(1)}, \qquad
U^{(1)} \cong \mu_{2^\infty}(K) \times \mathbb{Z}_2^{\,d},$$
the second isomorphism because $\log$ carries $U^{(m)}$ onto $\mathfrak{m}^m \cong (\mathcal{O},+)$
for $m > e/(p-1)$, so $U^{(1)}$ is a finitely generated $\mathbb{Z}_2$-module of rank $d$. Modulo
squares: $\pi^{\mathbb{Z}}$ gives $2$; $\mu_{q-1}$ has **odd** order and gives nothing;
$\mathbb{Z}_2^d$ gives $2^d$; $\mu_{2^\infty}(K) \ni -1$ gives $2$. Total $2^{d+2}$.

The factor $2^d = |2|_K^{-1}$ is the same one that governed $\#E(K)/mE(K)$ in §1, and the other two
are $\#H^0(K,\mu_2)$ and $\#H^2(K,\mu_2) = \#\mathrm{Br}(K)[2]$ — i.e. the count is the local Euler
characteristic $(\#H^0 \cdot \#H^2)/\#H^1 = \|2\|_K = 2^{-d}$ in one line.

For $\mathbb{Q}_4$: $d = 2$, $q = 4$, the odd part is $\mu_3$, and $\mu_{2^\infty}(\mathbb{Q}_4) =
\{\pm1\}$ since $i \notin \mathbb{Q}_4$ ($\mathbb{Q}_2(i)$ is *ramified*). So
$\mathbb{Q}_4^\times \cong 2^{\mathbb{Z}} \times \mu_3 \times \{\pm1\} \times \mathbb{Z}_2^2$ and the
order is $2 \cdot 1 \cdot 2 \cdot 4 = 16$ — which also predicts the shape of a basis: one odd
valuation class, $[-1]$, and two unit classes.

A finite check of the same count, with no structure theory: $(1+4a)^2 = 1 + 8(a + 2a^2)$ and
$a \mapsto a + 2a^2$ is bijective on $\mathcal{O}$ by Hensel, so $1 + 8\mathcal{O}$ consists of
squares and a unit is a square as soon as it is one **mod 8**. Hence
$\mathcal{O}^\times/(\mathcal{O}^\times)^2 = (\mathcal{O}/8\mathcal{O})^\times/\text{squares}$, a
computation in a ring of $64$ elements:

In [4]:
def mul8(u, v):
    "product in O/8O = (Z/8)[w]/(w^2+w+1), elements written as pairs (a,b) = a + b*w"
    (a, b), (c, d) = u, v
    return ((a*c - b*d) % 8, (a*d + b*c - b*d) % 8)

units8   = [(a, b) for a in range(8) for b in range(8) if (a^2 - a*b + b^2) % 2 == 1]
squares8 = sorted(set(mul8(u, u) for u in units8))

print("#(O/8O)    =", 8^2)
print("#(O/8O)^*  =", len(units8))
print("squares    =", len(squares8), " namely", squares8, " = mu_3 * {1, 5}")
print("index      =", len(units8) // len(squares8), " = #O^*/(O^*)^2")
print("times the Z/2 from the valuation:  #Q_4^*/(Q_4^*)^2 =",
      2 * len(units8) // len(squares8))

#(O/8O)    = 64
#(O/8O)^*  = 48
squares    = 6  namely [(0, 1), (0, 5), (1, 0), (3, 3), (5, 0), (7, 7)]  = mu_3 * {1, 5}
index      = 8  = #O^*/(O^*)^2
times the Z/2 from the valuation:  #Q_4^*/(Q_4^*)^2 = 16


## 3. A basis of $\mathbb{Q}_4^\times/(\mathbb{Q}_4^\times)^2$, and coordinates

Symbolic bookkeeping happens in the exact field $\mathbb{Q}(\omega)$; squareness is decided in
`K2`. The basis is $B = (2,\ -1,\ -2-\omega,\ -3-\omega)$ — one class of odd valuation, the class
of $-1$, and the two classes that will turn out to be the Kummer image.

In [5]:
KK.<om> = CyclotomicField(3)         # exact home for everything symbolic; om = zeta_3

def to2(z):
    "image of z in Q_4 under om |-> w"
    a, b = z.list()
    return K2(a) + K2(b)*w

def nrm(z):
    "exact norm to Q of a + b*om"
    a, b = z.list()
    return a^2 - a*b + b^2

def is_sq(z):
    "is z a square in Q_4 ?"
    return to2(z).is_square()

B      = [KK(2), KK(-1), -2 - om, -3 - om]
Bnames = ["2", "-1", "-2-om", "-3-om"]

def rep(m):
    "the product of the basis elements selected by the bits of m"
    z = KK(1)
    for i in range(4):
        if (m >> i) & 1:
            z *= B[i]
    return z

def coords(z):
    "coordinates of the class of z in the basis B, found by testing squareness"
    for m in range(16):
        if is_sq(z * rep(m)):
            return tuple((m >> i) & 1 for i in range(4))
    raise ValueError("z is not in the span of B")

print("basis:", Bnames)
print()
print("  coords          representative     norm    square?")
for m in range(16):
    z = rep(m)
    c = tuple((m >> i) & 1 for i in range(4))
    print("  " + str(c) + "   " + str(z).rjust(17) + "   "
          + str(nrm(z)).rjust(5) + "    " + str(is_sq(z)))

basis: ['2', '-1', '-2-om', '-3-om']

  coords          representative     norm    square?
  (0, 0, 0, 0)                   1       1    True
  (1, 0, 0, 0)                   2       4    False
  (0, 1, 0, 0)                  -1       1    False
  (1, 1, 0, 0)                  -2       4    False
  (0, 0, 1, 0)             -om - 2       3    False
  (1, 0, 1, 0)           -2*om - 4      12    False
  (0, 1, 1, 0)              om + 2       3    False
  (1, 1, 1, 0)            2*om + 4      12    False
  (0, 0, 0, 1)             -om - 3       7    False
  (1, 0, 0, 1)           -2*om - 6      28    False
  (0, 1, 0, 1)              om + 3       7    False
  (1, 1, 0, 1)            2*om + 6      28    False
  (0, 0, 1, 1)            4*om + 5      21    False
  (1, 0, 1, 1)           8*om + 10      84    False
  (0, 1, 1, 1)           -4*om - 5      21    False
  (1, 1, 1, 1)          -8*om - 10      84    False


Only the empty product is a square, so the $16$ classes are pairwise distinct: $B$ really is an
$\mathbb{F}_2$-basis, and `coords` is well defined on classes.

## 4. The set $V$

$-7 \equiv 1 \pmod 8$, so $\sqrt{-7} \in \mathbb{Z}_2^\times$; Hensel pins the root down by three
binary digits, since for $f(Y) = Y^2+7$ one has $|f(3)|_2 = 1/16 < |f'(3)|_2^2 = 1/4$. Take the
root with $y \equiv 3 \pmod 8$.

The addition $(-2,y) + (-1,0) = (-4,-3y)$ is verified **exactly**, in the number field
$\mathbb{Q}(\sqrt{-7})$ — which embeds in $\mathbb{Q}_2$ precisely because $-7$ is a square there.

In [6]:
F.<r> = NumberField(X^2 + 7)                 # r = sqrt(-7)
EF = EllipticCurve(F, [0,0,0,0,1])
P, T = EF([-2, r]), EF([-1, 0])
print("P     =", P, "   T =", T, "   order of T:", T.order())
print("P + T =", P + T)
print("P + T == (-4, -3r) ?", P + T == EF([-4, -3*r]))
print("exact checks: (-2)^3+1 =", (-2)^3+1, "= r^2 =", r^2,
      ";  (-4)^3+1 =", (-4)^3+1, "= (-3r)^2 =", (-3*r)^2)

P     = (-2 : r : 1)    T = (-1 : 0 : 1)    order of T: 2
P + T = (-4 : -3*r : 1)
P + T == (-4, -3r) ? True
exact checks: (-2)^3+1 = -7 = r^2 = -7 ;  (-4)^3+1 = -63 = (-3r)^2 = -63


In [7]:
Z2 = Zp(2, 40)
y  = Z2(-7).sqrt()
if y.lift() % 8 != 3:
    y = -y                                   # the root that is 3 mod 8

print("y            =", y)
print("y^2 + 7      =", y^2 + 7)
print("y mod 8      =", y.lift() % 8, "   (this alone determines y, by Hensel)")
print("y mod 2^20   =", y.lift() % 2^20)
print("-3y mod 2^20 =", (-3*y).lift() % 2^20)
print()
print("V = { O,  (-1, 0),  (-2, y),  (-4, -3y) }")

y            = 1 + 2 + 2^3 + 2^6 + 2^8 + 2^9 + 2^10 + 2^11 + 2^12 + 2^13 + 2^16 + 2^17 + 2^18 + 2^21 + 2^22 + 2^24 + 2^25 + 2^29 + 2^30 + 2^34 + 2^35 + 2^36 + O(2^39)
y^2 + 7      = O(2^40)
y mod 8      = 3    (this alone determines y, by Hensel)
y mod 2^20   = 474955
-3y mod 2^20 = 672287

V = { O,  (-1, 0),  (-2, y),  (-4, -3y) }


**On precision.** Every $x$-coordinate in $V$ is *rational*, so the Kummer classes below are exact —
no truncation enters $\delta$ at all. The $2$-adic approximation is only ever used to name the point,
and even the sign of $y$ is immaterial, since $-P \equiv P$ modulo $2E(\mathbb{Q}_2)$ and $\delta$
sees only $x$.

## 5. The image

In [8]:
pts = [("O", None), ("T = (-1,0)", -1), ("P = (-2,y)", -2), ("P+T = (-4,-3y)", -4)]

print("point".rjust(16) + " " + "x".rjust(4) + "   "
      + "delta = (x-1) - om".rjust(20) + " " + "N".rjust(5) + " "
      + "N mod 8".rjust(8) + "   coords")
for name, x in pts:
    z = KK(1) if x is None else KK(x) - 1 - om
    print(name.rjust(16) + " " + ("-" if x is None else str(x)).rjust(4) + "   "
          + str(z).rjust(20) + " " + str(nrm(z)).rjust(5) + " "
          + str(ZZ(nrm(z)) % 8).rjust(8) + "   " + str(coords(z)))

           point    x     delta = (x-1) - om     N  N mod 8   coords
               O    -                      1     1        1   (0, 0, 0, 0)
      T = (-1,0)   -1                -om - 2     3        3   (0, 0, 1, 0)
      P = (-2,y)   -2                -om - 3     7        7   (0, 0, 0, 1)
  P+T = (-4,-3y)   -4                -om - 5    21        5   (0, 0, 1, 1)


**Theorem.** $V$ is a complete set of representatives for $E(\mathbb{Q}_2)/2E(\mathbb{Q}_2)$, and
$$\operatorname{Im}(\delta) = \{1,\ -2-\omega,\ -3-\omega,\ -5-\omega\}\cdot(\mathbb{Q}_4^\times)^2
= \big\langle [-2-\omega],\ [-3-\omega]\big\rangle \cong (\mathbb{Z}/2)^2 .$$

*Proof.* A square in $\mathbb{Q}_4$ has square norm in $\mathbb{Q}_2$. The four norms are
$1, 3, 7, 21 \equiv 1, 3, 7, 5 \pmod 8$, hence lie in four different classes of
$\mathbb{Q}_2^\times/(\mathbb{Q}_2^\times)^2$; so the four values of $\delta$ lie in four different
classes. As $\delta$ is injective on $E(\mathbb{Q}_2)/2E(\mathbb{Q}_2)$ and that group has order $4$,
the four points of $V$ represent its four cosets, and their $\delta$-values are the whole image.
$\blacksquare$

Note the values are just $(x-1) - \omega$ for $x = -1, -2, -4$, and that the norm map is *injective*
on the image, carrying it isomorphically onto $\mathbb{Z}_2^\times/(\mathbb{Z}_2^\times)^2$.

In [9]:
zT, zP, zPT = -2 - om, -3 - om, -5 - om

# delta is a homomorphism:  delta(T) * delta(P) = delta(P+T)
print("(-2-om)(-3-om)          =", zT*zP)
print("same class as -5-om ?   ", is_sq(zT*zP*zPT), "  [product of the two is a square]")
print("by hand: (5+4om)(-5-om) =", (5+4*om)*(-5-om), "= 21*om^2, and 21 = 5 mod 8 is a square")
print()
# torsion: (0,1) = 2*(0,-1) must die, and (2,3) must land in the class of T
print("delta(0,1) =", KK(0)-1-om, "  square ?", is_sq(KK(0)-1-om), "  [= om^2]")
print("delta(2,3) =", KK(2)-1-om, "  coords", coords(KK(2)-1-om),
      " vs T:", coords(zT), "  [(2,3) = T + 3-torsion]")

(-2-om)(-3-om)          = 4*om + 5
same class as -5-om ?    True   [product of the two is a square]
by hand: (5+4om)(-5-om) = -21*om - 21 = 21*om^2, and 21 = 5 mod 8 is a square

delta(0,1) = -om - 1   square ? True   [= om^2]
delta(2,3) = -om + 1   coords (0, 0, 1, 0)  vs T: (0, 0, 1, 0)   [(2,3) = T + 3-torsion]


## 6. An independent brute-force scan

Forget the theory: sample $x \in \mathbb{Q}$ (dense in $\mathbb{Q}_2$) with $x^3+1$ a square in
$\mathbb{Q}_2$ — every such $x$ gives a genuine point of $E(\mathbb{Q}_2)$ — and record the class of
$x - \zeta_6$. Exactly four classes should appear, the four above.

In [10]:
Q2 = Qp(2, 60)
seen, count = {}, 0

for k in range(7):                            # x of valuation 0, -2, -4, ..., -12
    for a in range(-400, 401):
        if k > 0 and a % 2 == 0:
            continue
        x = QQ(a) / 4^k
        v = x^3 + 1
        if v == 0 or not Q2(v).is_square():
            continue
        count += 1
        seen.setdefault(coords(KK(x) - 1 - om), x)

print("points sampled           :", count)
print("distinct classes reached :", len(seen))
for c in sorted(seen):
    print("   ", c, "  first seen at x =", seen[c])

points sampled           : 1034
distinct classes reached : 4
    (0, 0, 0, 0)   first seen at x = -400
    (0, 0, 0, 1)   first seen at x = -394
    (0, 0, 1, 0)   first seen at x = -398
    (0, 0, 1, 1)   first seen at x = -396


## 7. Where the image sits: duality, and a twist that is easy to get wrong

$\#H^1(\mathbb{Q}_2, E[2]) = 16$, so local Tate duality forces the image to be **maximal isotropic**
of order $4$. But the pairing that does this is *not* the Hilbert symbol of $\mathbb{Q}_4$: the Weil
pairing on $E[2] = \operatorname{Ind}\mu_2$ is the standard pairing of the induced module *composed
with the swap* of the two conjugate points, so under Shapiro the Tate pairing becomes the Frobenius
twist $(a,b) \mapsto (a, \sigma b)_{\mathbb{Q}_4}$.

Hilbert symbols of $\mathbb{Q}_4$ are computed in $\mathbb{Q}(\omega)$ at its prime above $2$, which
is inert — so that completion *is* $\mathbb{Q}_4$.

In [11]:
P2 = KK.primes_above(2)[0]
print("prime above 2 :", P2, "   e =", P2.ramification_index(), "  f =", P2.residue_class_degree())

def hs(u, v):
    "Hilbert symbol of Q_4"
    return KK.hilbert_symbol(u, v, P2)

def sig(z):
    "the Frobenius of Q_4/Q_2:  om |-> om^2"
    a, b = z.list()
    return a + b*om^2

im = [KK(1), -2 - om, -3 - om, -5 - om]
print("sigma(-2-om) =", sig(-2-om), "    sigma(-3-om) =", sig(-3-om))
print()
print("plain    ((u_i, u_j))       ="); show(matrix(4, 4, lambda i, j: hs(im[i], im[j])))
print("twisted  ((u_i, sigma u_j)) ="); show(matrix(4, 4, lambda i, j: hs(im[i], sig(im[j]))))

prime above 2 : Fractional ideal (2)    e = 1   f = 2
sigma(-2-om) = om - 1     sigma(-3-om) = om - 2

plain    ((u_i, u_j))       =


[ 1  1  1  1]
[ 1 -1 -1  1]
[ 1 -1 -1  1]
[ 1  1  1  1]

twisted  ((u_i, sigma u_j)) =


[1 1 1 1]
[1 1 1 1]
[1 1 1 1]
[1 1 1 1]

In [12]:
# the annihilator of the image under the twisted pairing
ann = [rep(m) for m in range(16)
       if hs(rep(m), sig(-2-om)) == 1 and hs(rep(m), sig(-3-om)) == 1]

print("twisted annihilator :", len(ann), "classes:", ann)
print("equals the image ?  ", sorted(coords(z) for z in ann) == sorted(coords(u) for u in im))
print()
# position relative to the unramified classes H_ur = <-1, -2-om, -3-om>, of order 8
print("all four classes are unit classes (even though the reduction at 2 is additive):",
      all(nrm(u).valuation(2) == 0 for u in im))
print("(-1, sigma(-2-om)) =", hs(-1, sig(-2-om)),
      "  so [-1] is NOT in the image, and Im(delta) = { z in H_ur : (z, sigma(-2-om)) = 1 }")

twisted annihilator : 4 classes: [1, -om - 2, -om - 3, 4*om + 5]
equals the image ?   True

all four classes are unit classes (even though the reduction at 2 is additive): True
(-1, sigma(-2-om)) = -1   so [-1] is NOT in the image, and Im(delta) = { z in H_ur : (z, sigma(-2-om)) = 1 }


The plain symbol is **not** trivial on the image — already $(u,u)_{\mathbb{Q}_4} = -1$ for
$u = -2-\omega$ — while the twisted one vanishes identically and its annihilator is the image itself.
So the image is maximal isotropic on the nose, and the swap is not bookkeeping: it is what makes
duality come out.

## Summary

$$V = \{\mathcal{O},\ (-1,0),\ (-2,y),\ (-4,-3y)\}, \qquad
\operatorname{Im}(\delta) = \{1,\ -2-\omega,\ -3-\omega,\ -5-\omega\} \cdot (\mathbb{Q}_4^\times)^2 .$$

- the image is $\langle [-2-\omega], [-3-\omega]\rangle$, the span of the last two vectors of the
  basis $(2,\,-1,\,-2-\omega,\,-3-\omega)$ of the $16$-element group
  $\mathbb{Q}_4^\times/(\mathbb{Q}_4^\times)^2$;
- the norm map sends it isomorphically onto $\mathbb{Z}_2^\times/(\mathbb{Z}_2^\times)^2$, which is
  what separates the four classes;
- it consists of **unit** classes although $E$ has additive reduction at $2$, sitting with index $2$
  in the $8$-element unramified subgroup — which is itself too big to be a local condition;
- it is maximal isotropic for the Frobenius-twisted Hilbert symbol, as local duality demands.

Choosing the other root $\overline{\zeta_6}$ conjugates everything: the image becomes
$\langle[\omega-1],[\omega-2]\rangle$, a different maximal isotropic subgroup. Nothing is ambiguous —
$\delta$ and its image are conjugated together the moment one renames $\omega$ as $\omega^2$.